# ITBS-VoGen — Colab Training

Google Colab (T4 GPU) で RVC 話者モデルを学習するためのノートブック。ローカル (Mac) では推論のみ行い、重い学習だけクラウドに逃がす運用。

本ノートブックは **複数データセットから複数モデルを一度に学習する**構成。`SPEAKER_CONFIGS` に学習対象を列挙すると、依存セットアップは1回だけ走り、学習・保存は各モデル分ループする。

## 事前準備

1. **Google Drive に学習データをアップロード**: `MyDrive/itbs-vogen/Train/` 以下に各データセットフォルダ（例: `set1-1/`, `set1-2/`, `set1-3/`）を配置する。
2. **ランタイム変更**: メニュー → ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ = **T4 GPU**。
3. 以下のセルを上から順に実行。

## 所要時間の目安

T4 で 1 モデルあたり 20〜40 分（29分データ / 200 epoch 前提）。3モデルなら **合計1.5〜2時間**。Colab 無料枠のセッションタイムアウトに注意。途中で切れた場合は `SPEAKER_CONFIGS` から完了済みを消して再開すれば中断点から進められる。


## 1. GPU 確認

In [ ]:
!nvidia-smi

## 2. Google Drive マウント

学習データ読み込みと成果物保存に使用。認証プロンプトが出るので承認する。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 学習設定

学習対象のデータセットを `SPEAKER_CONFIGS` に列挙する。各エントリは `(モデル名, Drive上のデータディレクトリ名)`。

`DRIVE_ROOT` 配下の想定構造:
```
itbs-vogen/
├── Train/
│   ├── set1-1/*.wav
│   ├── set1-2/*.wav
│   └── set1-3/*.wav
└── models/            ← 各モデルの .pth / .index がここに吐かれる
```


In [ ]:
import os
from pathlib import Path

# === 学習対象 ===
# (モデル名, Drive 上のデータディレクトリ名)
SPEAKER_CONFIGS = [
    ('itbs_set1_1', 'set1-1'),
    ('itbs_set1_2', 'set1-2'),
    ('itbs_set1_3', 'set1-3'),
]

# === 共通ハイパーパラメータ ===
TOTAL_EPOCHS = 200                 # 典型値: 150-300
SAVE_EVERY_EPOCH = 50              # 途中保存の粒度
BATCH_SIZE = 12                    # T4 なら 12、A100 なら 24+
SR = '48k'                         # 32k / 40k / 48k
VERSION = 'v2'

# === Drive 上のパス ===
DRIVE_ROOT = Path('/content/drive/MyDrive/itbs-vogen')
DRIVE_TRAIN_ROOT = DRIVE_ROOT / 'Train'
DRIVE_MODELS_ROOT = DRIVE_ROOT / 'models'

# === Colab 作業ディレクトリ ===
WORK_DIR = Path('/content/ITBS-VoGen')

# 事前チェック
DRIVE_MODELS_ROOT.mkdir(parents=True, exist_ok=True)
for name, subdir in SPEAKER_CONFIGS:
    src = DRIVE_TRAIN_ROOT / subdir
    assert src.exists(), f'Training data not found: {src}'
    wavs = list(src.glob('*.wav'))
    print(f'[{name}] {src} -> {len(wavs)} wav files')


## 4. リポジトリ取得

公開リポジトリとRVC submoduleを取得。

In [ ]:
%cd /content
![ -d /content/ITBS-VoGen ] || git clone --recurse-submodules https://github.com/Arimuri/ITBS-VoGen.git
%cd /content/ITBS-VoGen
!git pull
!git submodule update --init --recursive

## 5. Python 3.10 + 依存インストール

Colab のデフォルト Python は 3.12 だが、RVC のピン (`numba==0.56.4`, `fairseq==0.12.2`) が 3.12 非対応なので **Python 3.10 を apt で追加**し、以降は `python3.10 -m pip` で明示的に使う。

5-10分かかる。途中で「Gemini が pyproject.toml を直しましょうか?」という提案が出ても **拒否**すること（こちらで設計済みの制約を知らないので、勝手に依存を足すと RVC と衝突する）。万一改変されたら `!git checkout pyproject.toml` で戻せる。


In [ ]:
%cd /content/ITBS-VoGen

# Reset pyproject.toml in case something (Gemini, a failed edit, etc.) corrupted it.
!git checkout pyproject.toml

# Install Python 3.10 alongside Colab's default 3.12.
!apt-get update -q
!apt-get install -y -q python3.10 python3.10-dev python3.10-venv python3.10-distutils
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10 - --quiet

# RVC requires an old omegaconf whose metadata trips pip >= 24.1.
!python3.10 -m pip install --quiet 'pip<24.1'

# Install RVC + our package into the Python 3.10 environment.
!python3.10 -m pip install --quiet -r third_party/rvc/requirements.txt
!python3.10 -m pip install --quiet -e .

# Verify.
!python3.10 -c "import sys; print('python:', sys.version)"
!python3.10 -m itbs_vogen.cli --help


## 6. ベースモデル取得

HuBERT + RMVPE + pretrained_v2 (48k, F0あり) をDL。初回のみ。

In [ ]:
!bash scripts/download_models.sh


## 7. 学習データ配置

Drive の各データセットを Colab ローカル (`/content/ITBS-VoGen/Train/<dir>/`) にコピーする (Drive直読みは遅い)。


In [ ]:
import shutil

for _, subdir in SPEAKER_CONFIGS:
    src = DRIVE_TRAIN_ROOT / subdir
    dst = WORK_DIR / 'Train' / subdir
    dst.mkdir(parents=True, exist_ok=True)
    for wav in src.glob('*.wav'):
        target = dst / wav.name
        if not target.exists():
            shutil.copy(wav, target)
    print(f'[{subdir}] copied {len(list(dst.glob("*.wav")))} wavs to {dst}')


## 8. 学習実行 (ループ)

`SPEAKER_CONFIGS` の全エントリを順番に学習する。T4 で各モデル 20-40 分、合計で数時間。各モデル完成時に次セルで Drive に保存される前提のため、中断された場合でも完了済みのモデルは Drive に残る。


In [ ]:
import subprocess
import time

for name, subdir in SPEAKER_CONFIGS:
    local_train = WORK_DIR / 'Train' / subdir
    drive_out = DRIVE_MODELS_ROOT / name
    drive_out.mkdir(parents=True, exist_ok=True)

    if (drive_out / 'model.pth').exists() and (drive_out / 'model.index').exists():
        print(f'[{name}] already trained (found in Drive). Skipping.')
        continue

    print(f'\n===== training {name} from {local_train} =====')
    t0 = time.time()
    cmd = [
        'python3.10', '-u', '-m', 'itbs_vogen.cli', 'train',
        '-s', name,
        '-d', str(local_train),
        '--sr', SR,
        '--version', VERSION,
        '--epochs', str(TOTAL_EPOCHS),
        '--save-every', str(SAVE_EVERY_EPOCH),
        '--batch-size', str(BATCH_SIZE),
        '--n-proc', '4',
        '--device', 'cuda:0',
    ]
    # Stream output live and raise with context if the subprocess fails.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Training for {name} failed with exit {proc.returncode}. See output above.')

    local_model = WORK_DIR / 'models' / 'speakers' / name
    for fname in ('model.pth', 'model.index'):
        src = local_model / fname
        if src.exists():
            shutil.copy(src, drive_out / fname)
    elapsed = time.time() - t0
    print(f'[{name}] done in {elapsed/60:.1f} min, saved to {drive_out}')


## 9. 保存確認

全モデルが Drive に保存されているか確認。


In [ ]:
for name, _ in SPEAKER_CONFIGS:
    drive_out = DRIVE_MODELS_ROOT / name
    pth = drive_out / 'model.pth'
    idx = drive_out / 'model.index'
    pth_ok = pth.exists()
    idx_ok = idx.exists()
    pth_size = f'{pth.stat().st_size / 1e6:.1f}MB' if pth_ok else 'missing'
    idx_size = f'{idx.stat().st_size / 1e6:.1f}MB' if idx_ok else 'missing'
    print(f'[{name}] pth: {pth_size}  |  index: {idx_size}')


## 10. ローカル (Mac) で使う

Mac 側で以下を実行:

1. Google Drive から 各モデルの `model.pth` + `model.index` をダウンロード
2. ローカルリポジトリの `models/speakers/<モデル名>/` に配置:
   ```
   models/speakers/itbs_set1_1/model.pth
   models/speakers/itbs_set1_1/model.index
   models/speakers/itbs_set1_2/model.pth
   models/speakers/itbs_set1_2/model.index
   models/speakers/itbs_set1_3/model.pth
   models/speakers/itbs_set1_3/model.index
   ```
3. 推論（各モデルで同じ入力を処理して聴き比べる想定）:
   ```bash
   itbs-vogen infer inputs/test_HonestyVo.wav \
       -o outputs/test_HonestyVo_set1_1.wav -s itbs_set1_1
   itbs-vogen infer inputs/test_HonestyVo.wav \
       -o outputs/test_HonestyVo_set1_2.wav -s itbs_set1_2
   itbs-vogen infer inputs/test_HonestyVo.wav \
       -o outputs/test_HonestyVo_set1_3.wav -s itbs_set1_3
   ```

## トラブルシュート

- **OOM (CUDA out of memory)**: `BATCH_SIZE` を 8, 6, 4 と下げる。
- **学習途中でセッション切断**: 完了済みモデルは Drive に保存されているので、ノートブックを再度実行すると「8. 学習実行」セルが完了済みを自動スキップして残りから再開する。
- **品質が低い**: epoch数を増やす (300-500)、または学習データを追加して再学習。
